# LangGraph Module · Day 05 — Human-in-the-Loop Patterns

**Phase 1 · Module 1: LangGraph & LangChain Deep Dive** · ~2.0 hrs

### Today's tasks (from the plan)
- [ ] Add **`interrupt_before` / `interrupt_after`** to a LangGraph flow
- [ ] Build an **approval gate**: agent pauses → human confirms → resumes
- [ ] Test with a **financial approval scenario** (credit limit change)

> Builds on Day 03 (checkpointers/state) and Day 04 (tool-calling agents). HITL is how a Barclays agent asks a human before it does anything with money.

## 0. Setup

HITL is a **graph mechanism**, not a model feature — most of this notebook needs **no API key**. The one section that shows an LLM deciding to call a risky tool is guarded by `HAS_KEY`.

Install (if needed): `pip install langgraph langchain-google-genai python-dotenv`

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
HAS_KEY = bool(os.getenv("GOOGLE_API_KEY"))
print("langgraph human-in-the-loop notebook")
print("GOOGLE_API_KEY:", "loaded ✅" if HAS_KEY else "MISSING ❌ (only the model cell is skipped)")

langgraph human-in-the-loop notebook
GOOGLE_API_KEY: loaded ✅


## 1. Why human-in-the-loop

An autonomous agent that can change a customer's **credit limit**, move money, or delete records is a liability if it acts unsupervised. HITL puts a **checkpoint** in the graph: the agent runs up to a sensitive step, **pauses**, and waits for a human to *approve*, *reject*, or *edit* — then resumes exactly where it stopped.

Two ways to pause in LangGraph:

| Approach | How | When to use |
|----------|-----|-------------|
| **Static** | `interrupt_before` / `interrupt_after` at **compile** time | pause at a *known* node every time (e.g. always before `execute`) |
| **Dynamic** | `interrupt(payload)` **inside** a node + `Command(resume=...)` | pause *conditionally*, and hand the human a rich question/payload |

Both need one thing first: **a checkpointer**.

## 2. Interrupts require a checkpointer

To pause and later resume, the graph must **persist its state** — otherwise there's nothing to come back to. That's the Day-03 checkpointer (`InMemorySaver` here; `SqliteSaver`/`PostgresSaver` in production). Every run is tagged with a **`thread_id`** so resume targets the right paused conversation.

No checkpointer → no interrupts. Full stop.

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

class CreditState(TypedDict):
    customer: str
    amount: int          # requested new credit limit
    status: str

checkpointer = InMemorySaver()
print("checkpointer ready:", type(checkpointer).__name__)

checkpointer ready: InMemorySaver


## 3. Static pause — `interrupt_before`

The scenario: a two-node flow — **`request`** records a credit-limit change, **`execute`** actually applies it. We want a human to sign off **before** `execute` ever runs.

Pass **`interrupt_before=["execute"]`** at compile time. The graph runs `request`, then **stops before `execute`** and hands control back to you.

In [3]:
def request(state: CreditState) -> dict:
    print(f"   [request] {state['customer']} wants a new limit of £{state['amount']:,}")
    return {"status": "pending_approval"}

def execute(state: CreditState) -> dict:
    print(f"   [execute] applying new credit limit £{state['amount']:,} for {state['customer']}")
    return {"status": "applied"}

builder = StateGraph(CreditState)
builder.add_node("request", request)
builder.add_node("execute", execute)
builder.add_edge(START, "request")
builder.add_edge("request", "execute")
builder.add_edge("execute", END)

graph = builder.compile(checkpointer=checkpointer, interrupt_before=["execute"])
print(graph.get_graph().draw_ascii())

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
 +---------+   
 | request |   
 +---------+   
      *        
      *        
      *        
 +---------+   
 | execute |   
 +---------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [4]:
config = {"configurable": {"thread_id": "cust-42"}}

# First invoke: runs request, then PAUSES before execute
result = graph.invoke({"customer": "Acme Ltd", "amount": 50_000, "status": ""}, config)
print("\nreturned state:", result)
print("status is still:", result["status"], "-> execute has NOT run yet")

   [request] Acme Ltd wants a new limit of £50,000

returned state: {'customer': 'Acme Ltd', 'amount': 50000, 'status': 'pending_approval'}
status is still: pending_approval -> execute has NOT run yet


☝️ Notice `execute` never printed and `status` is still `pending_approval`. The graph froze **before** the sensitive node. Inspect *where* it paused with `get_state` — `.next` tells you which node is about to run.

In [5]:
snapshot = graph.get_state(config)
print("paused before node:", snapshot.next)     # ('execute',)
print("current amount    :", snapshot.values["amount"])

paused before node: ('execute',)
current amount    : 50000


### Approve — resume with `invoke(None, config)`

Passing **`None`** as input means *"don't add anything, just continue from where you paused."* The checkpointer reloads the frozen state and runs `execute`.

In [6]:
approved = graph.invoke(None, config)   # None = resume
print("\nfinal state:", approved)
print("status now :", approved["status"])

   [execute] applying new credit limit £50,000 for Acme Ltd

final state: {'customer': 'Acme Ltd', 'amount': 50000, 'status': 'applied'}
status now : applied


## 4. The human can *edit* state before resuming — `update_state`

Approval isn't only yes/no. A risk officer might say *"approve, but cap it at £30,000."* Before resuming, call **`update_state`** to patch the paused state — then resume, and `execute` runs on the **edited** value.

In [7]:
config2 = {"configurable": {"thread_id": "cust-99"}}
graph.invoke({"customer": "Beta Corp", "amount": 80_000, "status": ""}, config2)

print("requested amount:", graph.get_state(config2).values["amount"])

# Human overrides the amount down to a policy cap, THEN approves
graph.update_state(config2, {"amount": 30_000})
print("after human edit:", graph.get_state(config2).values["amount"])

final = graph.invoke(None, config2)
print("executed with   :", final["amount"], "| status:", final["status"])

   [request] Beta Corp wants a new limit of £80,000
requested amount: 80000
after human edit: 30000
   [execute] applying new credit limit £30,000 for Beta Corp
executed with   : 30000 | status: applied


☝️ `update_state` writes a new checkpoint as if a node had produced those values. This is the **modify** path of approve/reject/**edit** — powerful for compliance overrides.

## 5. Dynamic pause — `interrupt()` + `Command(resume=...)`

Static interrupts always pause at the same node. Often you only want to pause **when it's risky** (e.g. amount over a threshold), and you want to hand the human a **question**, not just a frozen graph.

`interrupt(payload)` does both: called *inside* a node it (a) surfaces `payload` to the caller and (b) pauses. You resume by invoking with **`Command(resume=<answer>)`** — and `interrupt()` returns that answer, continuing the node from that line.

In [8]:
from langgraph.types import interrupt, Command

class ApprovalState(TypedDict):
    customer: str
    amount: int
    decision: str

def approval_gate(state: ApprovalState) -> dict:
    # Pause and ASK the human. Payload is whatever the reviewer needs to decide.
    answer = interrupt({
        "action": "credit_limit_change",
        "customer": state["customer"],
        "requested": state["amount"],
        "question": f"Approve raising {state['customer']}'s limit to £{state['amount']:,}?",
    })
    # Execution resumes HERE with answer == whatever we passed to Command(resume=...)
    return {"decision": answer}

b = StateGraph(ApprovalState)
b.add_node("approval_gate", approval_gate)
b.add_edge(START, "approval_gate")
b.add_edge("approval_gate", END)
dyn = b.compile(checkpointer=InMemorySaver())
print("dynamic-interrupt graph compiled")

dynamic-interrupt graph compiled


In [9]:
cfg = {"configurable": {"thread_id": "dyn-1"}}
paused = dyn.invoke({"customer": "Gamma Inc", "amount": 120_000, "decision": ""}, cfg)

# The interrupt payload surfaces under the __interrupt__ key
intr = paused["__interrupt__"][0]
print("PAUSED. Human is asked:")
print("  ", intr.value["question"])

PAUSED. Human is asked:
   Approve raising Gamma Inc's limit to £120,000?


In [10]:
# Human answers -> resume with Command(resume=...)
resumed = dyn.invoke(Command(resume="approved"), cfg)
print("decision recorded:", resumed["decision"])

decision recorded: approved


## 6. Approve **or reject** — branch on the human's answer

A real gate has two exits. Put the `interrupt()` in a gate node, then use a **conditional edge** to route: approved → `execute`, rejected → `notify_declined`. The human's word decides the path.

In [11]:
class GateState(TypedDict):
    customer: str
    amount: int
    decision: str
    status: str

def gate(state: GateState) -> dict:
    answer = interrupt({"question": f"Approve £{state['amount']:,} limit for {state['customer']}?"})
    return {"decision": answer}

def do_execute(state: GateState) -> dict:
    return {"status": f"APPLIED £{state['amount']:,} for {state['customer']}"}

def declined(state: GateState) -> dict:
    return {"status": f"DECLINED limit change for {state['customer']}"}

def route(state: GateState) -> str:
    return "do_execute" if state["decision"] == "approve" else "declined"

gb = StateGraph(GateState)
gb.add_node("gate", gate)
gb.add_node("do_execute", do_execute)
gb.add_node("declined", declined)
gb.add_edge(START, "gate")
gb.add_conditional_edges("gate", route, {"do_execute": "do_execute", "declined": "declined"})
gb.add_edge("do_execute", END)
gb.add_edge("declined", END)
gate_graph = gb.compile(checkpointer=InMemorySaver())
print(gate_graph.get_graph().draw_ascii())

          +-----------+             
          | __start__ |             
          +-----------+             
                *                   
                *                   
                *                   
            +------+                
            | gate |                
            +------+.               
           ..        ..             
         ..            ..           
        .                .          
+----------+        +------------+  
| declined |        | do_execute |  
+----------+        +------------+  
           **        **             
             **    **               
               *  *                 
           +---------+              
           | __end__ |              
           +---------+              


In [12]:
def run(thread, amount, human_answer):
    cfg = {"configurable": {"thread_id": thread}}
    p = gate_graph.invoke({"customer": "Delta LLC", "amount": amount, "decision": "", "status": ""}, cfg)
    print("asked:", p["__interrupt__"][0].value["question"])
    out = gate_graph.invoke(Command(resume=human_answer), cfg)
    print(f"human said {human_answer!r} -> {out['status']}\n")

run("g-approve", 15_000, "approve")
run("g-reject", 500_000, "reject")

asked: Approve £15,000 limit for Delta LLC?
human said 'approve' -> APPLIED £15,000 for Delta LLC

asked: Approve £500,000 limit for Delta LLC?
human said 'reject' -> DECLINED limit change for Delta LLC



☝️ Same gate, two outcomes, decided entirely by the human's `Command(resume=...)`. This approve/reject fork is the **core banking HITL pattern** — everything else is payload and policy on top.

## 7. Full scenario — an LLM agent that must ask before changing a limit

Tie it together: an agent gets a request, but the **policy** is *any change over £50,000 needs human sign-off*. The gate only interrupts when the threshold is crossed — small changes auto-approve, big ones pause for a human.

The `HAS_KEY` block lets a real model draft the customer-facing message after approval; without a key we skip only that flourish.

In [13]:
THRESHOLD = 50_000

class BankState(TypedDict):
    customer: str
    amount: int
    decision: str
    status: str
    message: str

def policy_gate(state: BankState) -> dict:
    if state["amount"] <= THRESHOLD:
        return {"decision": "auto_approved"}          # under limit: no human needed
    answer = interrupt({                              # over limit: pause for human
        "risk": "HIGH",
        "question": f"£{state['amount']:,} exceeds £{THRESHOLD:,} auto-approve cap. Approve for {state['customer']}?",
    })
    return {"decision": answer}

def apply_change(state: BankState) -> dict:
    if state["decision"] in ("auto_approved", "approve"):
        return {"status": f"APPLIED £{state['amount']:,}"}
    return {"status": "DECLINED"}

def route(state: BankState) -> str:
    return "apply_change"

bank = StateGraph(BankState)
bank.add_node("policy_gate", policy_gate)
bank.add_node("apply_change", apply_change)
bank.add_edge(START, "policy_gate")
bank.add_edge("policy_gate", "apply_change")
bank.add_edge("apply_change", END)
bank_graph = bank.compile(checkpointer=InMemorySaver())
print("banking HITL agent compiled | auto-approve cap =", f"£{THRESHOLD:,}")

banking HITL agent compiled | auto-approve cap = £50,000


In [14]:
# Case A: small change -> auto-approved, no interrupt
cfgA = {"configurable": {"thread_id": "bank-small"}}
outA = bank_graph.invoke({"customer": "Epsilon", "amount": 10_000, "decision": "", "status": "", "message": ""}, cfgA)
print("small change ->", outA["decision"], "|", outA["status"])
print("interrupted? ", "__interrupt__" in outA, "\n")

# Case B: large change -> pauses for human
cfgB = {"configurable": {"thread_id": "bank-large"}}
outB = bank_graph.invoke({"customer": "Zeta PLC", "amount": 250_000, "decision": "", "status": "", "message": ""}, cfgB)
print("large change -> PAUSED. risk:", outB["__interrupt__"][0].value["risk"])
print("  ", outB["__interrupt__"][0].value["question"])
finalB = bank_graph.invoke(Command(resume="approve"), cfgB)
print("human approved ->", finalB["status"])

small change -> auto_approved | APPLIED £10,000
interrupted?  False 

large change -> PAUSED. risk: HIGH
   £250,000 exceeds £50,000 auto-approve cap. Approve for Zeta PLC?
human approved -> APPLIED £250,000


In [15]:
if HAS_KEY:
    from langchain_google_genai import ChatGoogleGenerativeAI
    model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
    prompt = (f"In one short sentence, tell customer {finalB['customer']} their credit limit "
              f"was approved and set to £{finalB['amount']:,}. Professional banking tone.")
    print("customer message:\n ", model.invoke(prompt).content)
else:
    print("skipped (no API key) - the graph result above is the real deliverable")

customer message:
  Your credit limit has been approved and set at £250,000.


## 8. Recap

| Concept | What it is | Why it's used |
|---------|-----------|---------------|
| **checkpointer** | persists state per `thread_id` | prerequisite — no persistence, no pause/resume |
| **`interrupt_before` / `interrupt_after`** | static pause at compile time | always stop at a known sensitive node |
| **`invoke(None, config)`** | resume from a static interrupt | continue after human approves |
| **`get_state` / `.next`** | inspect the paused graph | see *where* and *on what data* it stopped |
| **`update_state`** | patch state before resuming | the **edit/override** approval path |
| **`interrupt(payload)`** | dynamic pause + ask a question | conditional gates with a human-readable payload |
| **`Command(resume=...)`** | resume a dynamic interrupt | feed the human's answer back into the node |
| **conditional edge on decision** | approve → execute, reject → decline | the core approve/reject banking fork |

